<a href="https://colab.research.google.com/github/bekadiscoteka/ChatBot-for-slang/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22TB_Transformer_Colab_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🫁 Transformer Architectures for Tuberculosis Classification
### Initial Experimental Implementation

This notebook implements a simple end-to-end pipeline for TB classification
from chest X-rays using Vision Transformer (ViT), Swin Transformer, and DeiT.

**Datasets used:** Montgomery County + Shenzhen Hospital (public NLM/NIH datasets)

**Pipeline:**
1. Install dependencies
2. Download datasets from Kaggle
3. Preprocess & augment images
4. Fine-tune transformer models
5. Evaluate & compare results
6. Visualise attention maps

> ⚡ **Runtime:** Set to `GPU` → `T4` in *Runtime > Change runtime type*

## 1. Install Dependencies

In [ ]:
!pip install -q timm kaggle torchmetrics matplotlib seaborn scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.2 MB/s eta 0:00:00


## 2. Imports & Reproducibility

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


## 3. Download Dataset from Kaggle

Upload your `kaggle.json` API key, or use the cell below to authenticate.

> Get your key from: https://www.kaggle.com/settings → *Create New Token*

In [ ]:
# ── Option A: upload kaggle.json manually ────────────────────
from google.colab import files
print('Upload your kaggle.json file:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Upload your kaggle.json file:


KeyboardInterrupt: 

In [ ]:
# Download TB chest X-ray dataset (Montgomery + Shenzhen combined)
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset
!unzip -q tuberculosis-tb-chest-xray-dataset.zip -d /content/tb_data

# Inspect structure
!find /content/tb_data -type d | head -20

## 4. Build Dataset & DataLoaders

In [ ]:
# ── Config ───────────────────────────────────────────────────
DATA_ROOT  = Path('/content/tb_data')
IMG_SIZE   = 224          # ViT / DeiT / Swin-T default
BATCH_SIZE = 16
NUM_EPOCHS = 20           # keep low for initial experiments
LR         = 1e-4
NUM_CLASSES = 2           # TB / Normal

# ── Locate images ────────────────────────────────────────────
def collect_image_paths(root: Path):
    """Walk the dataset folder and return (path, label) pairs."""
    data = []
    for label_name, label_idx in [('Normal', 0), ('Tuberculosis', 1)]:
        folder = root / label_name
        if not folder.exists():
            # try lowercase / alternative names
            for alt in ['normal', 'tb', 'TB', 'TUBERCULOSIS']:
                alt_folder = root / alt
                if alt_folder.exists():
                    folder = alt_folder
                    break
        for img_path in folder.glob('**/*.png'):
            data.append((str(img_path), label_idx))
        for img_path in folder.glob('**/*.jpg'):
            data.append((str(img_path), label_idx))
    return data

all_data = collect_image_paths(DATA_ROOT)
print(f'Total images found: {len(all_data)}')
labels   = [d[1] for d in all_data]
print(f'Normal: {labels.count(0)}  |  TB: {labels.count(1)}')

In [ ]:
# ── Stratified train / val / test split ──────────────────────
train_data, temp_data = train_test_split(
    all_data, test_size=0.30, stratify=labels, random_state=SEED)

temp_labels = [d[1] for d in temp_data]
val_data, test_data = train_test_split(
    temp_data, test_size=0.50, stratify=temp_labels, random_state=SEED)

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

In [ ]:
# ── Transforms ───────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomErasing(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# ── Dataset class ────────────────────────────────────────────
class TBDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data      = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


# ── DataLoaders ───────────────────────────────────────────────
train_loader = DataLoader(TBDataset(train_data, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(TBDataset(val_data,   eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(TBDataset(test_data,  eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('DataLoaders ready.')

In [ ]:
# ── Quick visual sanity check ────────────────────────────────
imgs, lbls = next(iter(train_loader))
label_names = ['Normal', 'Tuberculosis']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)  # denormalise
    img = np.clip(img, 0, 1)
    ax.imshow(img, cmap='gray')
    ax.set_title(label_names[lbls[i].item()], fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Training Images', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Model Factory

We use `timm` to load three transformer models with ImageNet-21k pre-trained weights.
Only the classification head is replaced for binary TB/Normal output.

In [ ]:
def build_model(model_name: str, num_classes: int = 2, pretrained: bool = True):
    """
    model_name options:
      'vit_base_patch16_224'          -> ViT-B/16
      'swin_tiny_patch4_window7_224'  -> Swin-T
      'deit_base_patch16_224'         -> DeiT-B
    """
    model = timm.create_model(model_name, pretrained=pretrained,
                               num_classes=num_classes)
    return model.to(DEVICE)


# ── Choose model to experiment with ──────────────────────────
# Swap MODEL_NAME to switch architecture
MODEL_NAME = 'vit_base_patch16_224'    # ← change here
# MODEL_NAME = 'swin_tiny_patch4_window7_224'
# MODEL_NAME = 'deit_base_patch16_224'

model = build_model(MODEL_NAME)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model : {MODEL_NAME}')
print(f'Params: {total_params:.1f} M')

## 6. Training Loop

In [ ]:
# ── Loss, optimiser, scheduler ───────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


# ── Train / eval helpers ─────────────────────────────────────
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            if train:
                optimizer.zero_grad()

            outputs = model(imgs)
            loss    = criterion(outputs, labels)

            if train:
                loss.backward()
                optimizer.step()

            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            preds = outputs.argmax(dim=1)

            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total
    accuracy = correct / total
    auc      = roc_auc_score(all_labels, all_probs)
    return avg_loss, accuracy, auc


# ── Training loop ─────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [],
           'train_acc':  [], 'val_acc':  [],
           'train_auc':  [], 'val_auc':  []}

best_val_loss = float('inf')
best_ckpt     = '/content/best_model.pth'
patience, patience_counter = 7, 0

print(f'\nTraining {MODEL_NAME} for {NUM_EPOCHS} epochs...\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val Loss":>9} | '
      f'{"Train Acc":>9} | {"Val Acc":>8} | {"Val AUC":>8}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, tr_auc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc, vl_auc = run_epoch(val_loader,   train=False)
    scheduler.step()

    for k, v in zip(['train_loss','val_loss','train_acc','val_acc','train_auc','val_auc'],
                    [tr_loss, vl_loss, tr_acc, vl_acc, tr_auc, vl_auc]):
        history[k].append(v)

    marker = ''
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), best_ckpt)
        patience_counter = 0
        marker = ' ✓'
    else:
        patience_counter += 1

    print(f'{epoch:>6} | {tr_loss:>10.4f} | {vl_loss:>9.4f} | '
          f'{tr_acc:>9.4f} | {vl_acc:>8.4f} | {vl_auc:>8.4f}{marker}')

    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest checkpoint saved to: {best_ckpt}')

## 7. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
metrics   = [('Loss', 'train_loss', 'val_loss'),
             ('Accuracy', 'train_acc', 'val_acc'),
             ('AUC-ROC',  'train_auc', 'val_auc')]

for ax, (title, tr_key, vl_key) in zip(axes, metrics):
    ax.plot(epochs_ran, history[tr_key], label='Train', linewidth=2)
    ax.plot(epochs_ran, history[vl_key], label='Val',   linewidth=2, linestyle='--')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle(f'Training Curves — {MODEL_NAME}', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Test Set Evaluation

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs   = imgs.to(DEVICE)
        out    = model(imgs)
        probs  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds  = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

# ── Metrics ───────────────────────────────────────────────────
print('=' * 55)
print(f'  Test Results — {MODEL_NAME}')
print('=' * 55)
print(classification_report(all_labels, all_preds,
                             target_names=['Normal', 'Tuberculosis']))
print(f'  AUC-ROC : {roc_auc_score(all_labels, all_probs):.4f}')
print('=' * 55)

In [ ]:
# ── Confusion matrix + ROC curve ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'TB'], yticklabels=['Normal', 'TB'])
axes[0].set_title('Confusion Matrix', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score   = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, linewidth=2.5, label=f'AUC = {auc_score:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_title('ROC Curve', fontsize=12)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.suptitle(f'Evaluation — {MODEL_NAME}', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Attention Map Visualisation (ViT)

We use **Attention Rollout** to propagate attention weights through all
transformer layers and overlay the result on the original X-ray image.

> ⚠️ This cell works directly for ViT and DeiT.
> For Swin Transformer, window-level attention aggregation differs slightly.

In [ ]:
import torch.nn.functional as F
import cv2

def get_attention_rollout(model, img_tensor):
    """Compute attention rollout for a ViT / DeiT model."""
    attn_weights = []

    # Register hooks on every attention block
    hooks = []
    for block in model.blocks:
        hooks.append(
            block.attn.register_forward_hook(
                lambda m, inp, out: attn_weights.append(
                    m.attn_drop.p if hasattr(m, 'attn_drop') else None
                )
            )
        )

    # ─── Simpler approach: extract CLS attention from last block ───
    attn_maps = []
    def save_attn(module, inp, output):
        # output shape for timm ViT: (B, heads, N, N)
        attn_maps.append(output.detach().cpu())

    for hook in hooks:
        hook.remove()

    # Use the built-in attention getter in timm
    model.eval()
    with torch.no_grad():
        _ = model(img_tensor.unsqueeze(0).to(DEVICE))

    return attn_maps


def visualise_attention(model, dataset, idx=0):
    """Overlay a simple CLS-token attention map on the original image."""
    img_tensor, label = dataset[idx]
    label_names = ['Normal', 'Tuberculosis']

    # ── Collect last-layer attention via hooks ──────────────────
    attn_store = []
    def hook_fn(module, inp, out):
        # timm stores attention weights inside the attn module
        if hasattr(module, 'attn_drop'):
            pass  # not needed
        attn_store.append(out)

    # Access last transformer block's attention weights
    last_block = list(model.blocks)[-1]

    # Patch to capture attention
    original_forward = last_block.attn.forward
    captured = {}

    def patched_attn_forward(x):
        B, N, C = x.shape
        qkv   = last_block.attn.qkv(x).reshape(
                    B, N, 3, last_block.attn.num_heads,
                    C // last_block.attn.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scale = last_block.attn.scale
        attn  = (q @ k.transpose(-2, -1)) * scale
        attn  = attn.softmax(dim=-1)
        captured['attn'] = attn.detach().cpu()   # (B, heads, N, N)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = last_block.attn.proj(x)
        return x

    last_block.attn.forward = patched_attn_forward

    model.eval()
    with torch.no_grad():
        logits = model(img_tensor.unsqueeze(0).to(DEVICE))
        pred   = logits.argmax(dim=1).item()

    last_block.attn.forward = original_forward   # restore

    # ── Build attention map from CLS token (token 0) ────────────
    attn = captured['attn'][0]          # (heads, N, N)
    cls_attn = attn.mean(0)[0, 1:]      # avg heads, CLS row, drop CLS col
    n_patches = int(cls_attn.shape[0] ** 0.5)
    cls_attn  = cls_attn.reshape(n_patches, n_patches).numpy()
    cls_attn  = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)

    # Upsample to image size
    attn_map = cv2.resize(cls_attn, (IMG_SIZE, IMG_SIZE))
    attn_map = cv2.applyColorMap(
        np.uint8(255 * attn_map), cv2.COLORMAP_JET)
    attn_map = cv2.cvtColor(attn_map, cv2.COLOR_BGR2RGB)

    # Denormalise original image
    orig = img_tensor.permute(1, 2, 0).numpy()
    orig = orig * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    orig = np.clip(orig, 0, 1)
    orig_uint8 = np.uint8(255 * orig)

    overlay = cv2.addWeighted(orig_uint8, 0.6, attn_map, 0.4, 0)

    # ── Plot ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(orig_uint8)
    axes[0].set_title(f'Original CXR\nTrue: {label_names[label]}', fontsize=11)
    axes[1].imshow(cls_attn, cmap='hot')
    axes[1].set_title('Attention Map (CLS)', fontsize=11)
    axes[2].imshow(overlay)
    axes[2].set_title(f'Overlay\nPred: {label_names[pred]}', fontsize=11)
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'Attention Rollout — {MODEL_NAME}', fontsize=12)
    plt.tight_layout()
    plt.show()


# ── Visualise a few test samples ─────────────────────────────
test_dataset = TBDataset(test_data, eval_tf)
for sample_idx in [0, 5, 10]:
    visualise_attention(model, test_dataset, idx=sample_idx)

## 10. Compare All Three Models

Run this section to train all three architectures and produce a summary table.

In [ ]:
def train_and_evaluate(model_name, epochs=NUM_EPOCHS):
    """Full train + test pipeline for a given timm model name."""
    print(f'\n{"="*55}')
    print(f'  Training: {model_name}')
    print(f'{"="*55}')

    net       = build_model(model_name)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ckpt_path = f'/content/best_{model_name.split("/")[-1]}.pth'

    best_val, patience_c = float('inf'), 0
    PATIENCE = 7

    for epoch in range(1, epochs + 1):
        # -- train --
        net.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(net(imgs), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # -- val loss (for early stopping) --
        net.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                val_loss += criterion(net(imgs), labels).item()

        if val_loss < best_val:
            best_val = val_loss
            torch.save(net.state_dict(), ckpt_path)
            patience_c = 0
        else:
            patience_c += 1

        if epoch % 5 == 0:
            print(f'  Epoch {epoch:>3} | Val loss: {val_loss:.4f}')

        if patience_c >= PATIENCE:
            print(f'  Early stop at epoch {epoch}')
            break

    # -- test --
    net.load_state_dict(torch.load(ckpt_path))
    net.eval()
    preds_list, labels_list, probs_list = [], [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            out = net(imgs.to(DEVICE))
            probs_list.extend(torch.softmax(out, 1)[:, 1].cpu().numpy())
            preds_list.extend(out.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())

    from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
    results = {
        'Model':     model_name,
        'Accuracy':  round(accuracy_score(labels_list, preds_list), 4),
        'Precision': round(precision_score(labels_list, preds_list), 4),
        'Recall':    round(recall_score(labels_list, preds_list), 4),
        'F1':        round(f1_score(labels_list, preds_list), 4),
        'AUC':       round(roc_auc_score(labels_list, probs_list), 4),
    }
    print(f'  Done. AUC = {results["AUC"]}')
    return results


# ── Run all three ─────────────────────────────────────────────
MODELS = [
    'vit_base_patch16_224',
    'swin_tiny_patch4_window7_224',
    'deit_base_patch16_224',
]

results_list = [train_and_evaluate(m, epochs=15) for m in MODELS]

# ── Summary table ─────────────────────────────────────────────
df_results = pd.DataFrame(results_list).set_index('Model')
print('\n', df_results.to_string())

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────
df_results.plot(kind='bar', figsize=(12, 5), ylim=(0.7, 1.0),
                rot=15, colormap='tab10', edgecolor='black', linewidth=0.5)
plt.title('Model Comparison on TB Test Set', fontsize=13)
plt.ylabel('Score')
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Save Results & Download

Save the comparison table as a CSV and download everything.

In [ ]:
df_results.to_csv('/content/tb_transformer_results.csv')
print('Saved: /content/tb_transformer_results.csv')

# Download results CSV
from google.colab import files
files.download('/content/tb_transformer_results.csv')

---
## 📌 Quick Reference

| Model | timm name | Params |
|---|---|---|
| ViT-B/16 | `vit_base_patch16_224` | ~86M |
| Swin-T | `swin_tiny_patch4_window7_224` | ~28M |
| DeiT-B | `deit_base_patch16_224` | ~86M |

**Key hyperparameters to tune next:**
- `LR` — try `5e-5` or `2e-4`
- `IMG_SIZE` — try `384` for Swin / ViT (needs more VRAM)
- `NUM_EPOCHS` — increase to 50 with early stopping for full paper results
- Pre-training source — try `pretrained_cfg='in21k'` in `timm.create_model`

**Next steps for the paper:**
1. Run 5× with different seeds and report mean ± std
2. Add CheXpert pre-training ablation
3. Extend attention visualisation to Swin using window-level aggregation
4. Add class-imbalance handling (weighted loss or oversampling)